In [1]:

#This is not finished yet, I have a lot I would like to explore with the SRILM but it is what I have done so far :) 


In [2]:
#Function taken from Assignment 1

import subprocess
import sys
import os

def run_srilm(command, warnings=False):
    """Runs the provided SRILM command and prints any resulting messages
    
    Arguments
    ---------
    command: str; the SRILM command, as it would be entered in the Terminal
    warnings: bool; whether or not to print warnings alongside output
    """
    if sys.platform == 'win32':
        # This is just so that it works on my Windows machine.
        # For the purposes of this class, just ignore this line!
        command = [r"C:\cygwin64\bin\bash.exe ", "--login", "-c",
            "source ~/.bashrc && cd / && /srilm/bin/cygwin64/" + command]

    stdout = subprocess.PIPE
    if warnings:
        stderr = subprocess.STDOUT
    else:
        stderr = subprocess.PIPE
        
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=stdout, stderr=stderr)
        output_msg = result.stdout
        if output_msg:
            print(output_msg.decode('UTF-8'))
        else:
            print("The command ran successfully")
            
    except subprocess.CalledProcessError as e:
        print("The command did not run, and produced the following error message:\n{}".format(e.stderr.decode('UTF-8')))

In [3]:
#Import the SRILM-ready data set made in the preprocessing nb
from pathlib import Path

file_path = Path("../output/countries/")
txt_files = sorted(file_path.glob("*.txt"))

print(f"Imported {len(txt_files)} files:")
print([f.name for f in txt_files])

Imported 8 files:
['CZ.txt', 'DE.txt', 'GR.txt', 'NL.txt', 'PL.txt', 'PO.txt', 'SP.txt', 'UK.txt']


In [4]:
#split into train and test files using 80/20 split
import random

input_directory = Path("../output/countries")
train_directory = Path("../output/train")
test_directory = Path("../output/test")

train_directory.mkdir(parents=True, exist_ok=True)
test_directory.mkdir(parents=True, exist_ok=True)

random.seed(2026)
start = 0
skipped = []

for file_path in input_directory.glob("*.txt"):
    nation = file_path.stem
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    #shuffle the speeches so there is no issues with the time of the speech impacting results
    random.shuffle(lines)

    if len(lines) > 2000:
        
        
        split_idx = int(len(lines) * 0.8)
        train_lines = lines[:split_idx]
        test_lines = lines[split_idx:]
    
        #making the training sets
        train_file = train_directory / f"{nation}_train.txt" 
        with open(train_file, "w", encoding="utf-8") as f:
            f.write("\n".join(train_lines))
    
        #making the test sets
        test_file = test_directory / f"{nation}_test.txt"
        with open(test_file, "w", encoding="utf-8") as f:
            f.write("\n".join(test_lines))
    
    
        if start == 0:
            print("The splits are:")
            print("-" * 30)
            start += 1
        print(f"{nation}: {len(train_lines)} train / {len(test_lines)} test")

    else:
        skipped.append(nation)
        pass


The splits are:
------------------------------
SP: 7242 train / 1811 test
NL: 33656 train / 8414 test
DE: 3948 train / 987 test
GR: 34904 train / 8727 test
UK: 95357 train / 23840 test
PL: 12380 train / 3095 test


In [5]:
#make the models folder for SRILM to input made models into

model_directory = Path("../output/models")
model_directory.mkdir(parents=True, exist_ok=True)



In [6]:
#bigrams

for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"Trigram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 2 -lm ../output/models/{name}_trigram.lm -kndiscount")
        run_srilm(f"ngram -order 2 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")

Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 806 OOVs
0 zeroprobs, logprob= -64672.63 ppl= 145.9469 ppl1= 173.0283

Trigram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 1590 OOVs
0 zeroprobs, logprob= -474161.8 ppl= 172.8914 ppl1= 215.7239

Trigram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 221 OOVs
0 zeroprobs, logprob= -514327.7 ppl= 100.4885 ppl1= 117.466

Trigram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 1315 OOVs
0 zeroprobs, logprob= -131961 ppl= 210.2845 ppl1= 286.1821

Trigram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 1163 OOVs
0 zeroprobs, logprob=

In [7]:
#Trigrams
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"Trigram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 3 -lm ../output/models/{name}_trigram.lm -kndiscount")
        run_srilm(f"ngram -order 3 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")


Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 806 OOVs
0 zeroprobs, logprob= -63297.84 ppl= 131.2771 ppl1= 155.0743

Trigram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 1590 OOVs
0 zeroprobs, logprob= -458615.6 ppl= 146.0173 ppl1= 180.8747

Trigram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 221 OOVs
0 zeroprobs, logprob= -379469.9 ppl= 30.00249 ppl1= 33.66487

Trigram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 1315 OOVs
0 zeroprobs, logprob= -128992.2 ppl= 186.4443 ppl1= 251.9842

Trigram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 1163 OOVs
0 zeroprobs, logpr

In [10]:
#fivegram 
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"5-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 5 -lm ../output/models/{name}_fivegram.lm -kndiscount")
        run_srilm(f"ngram -order 5 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fivegram.lm")


5-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 806 OOVs
0 zeroprobs, logprob= -63428.64 ppl= 132.6068 ppl1= 156.699

5-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 1590 OOVs
0 zeroprobs, logprob= -458257 ppl= 145.4494 ppl1= 180.141

5-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 221 OOVs
0 zeroprobs, logprob= -276676.8 ppl= 11.94032 ppl1= 12.98632

5-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 1315 OOVs
0 zeroprobs, logprob= -128977.5 ppl= 186.3337 ppl1= 251.8261

5-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 1163 OOVs
0 zeroprobs, logprob= -9892

In [15]:
#fourgram model (WINNER)

print("KND UNINT")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"4-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram.lm -kndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram.lm")


KND UNINT
4-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 0 OOVs
0 zeroprobs, logprob= -64902.2 ppl= 130.2674 ppl1= 153.1483

4-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 0 OOVs
0 zeroprobs, logprob= -461629.1 ppl= 145.345 ppl1= 179.7074

4-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 0 OOVs
0 zeroprobs, logprob= -301779.2 ppl= 14.91839 ppl1= 16.34675

4-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 0 OOVs
0 zeroprobs, logprob= -131509.8 ppl= 183.0184 ppl1= 245.3238

4-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 0 OOVs
0 zeroprobs, logprob= -10108

In [14]:
#Smoothing methods

#KND INT
print("KND INT)")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"4-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_INT.lm -kndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_INT.lm")


KND INT)
4-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 0 OOVs
0 zeroprobs, logprob= -66427.1 ppl= 146.0578 ppl1= 172.3662

4-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 0 OOVs
0 zeroprobs, logprob= -461161.1 ppl= 144.613 ppl1= 178.7639

4-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 0 OOVs
0 zeroprobs, logprob= -276497.5 ppl= 11.89577 ppl1= 12.93527

4-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 0 OOVs
0 zeroprobs, logprob= -134197.4 ppl= 203.5786 ppl1= 274.5223

4-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 0 OOVs
0 zeroprobs, logprob= -103496

In [16]:
#Modified KN Discount W/o interpolation


print("Modified KND)")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"{name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd.lm -ukndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd.lm")


Modified KND)
DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 0 OOVs
0 zeroprobs, logprob= -64237.73 ppl= 123.9322 ppl1= 145.4591

GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 0 OOVs
0 zeroprobs, logprob= -457910.3 ppl= 139.6303 ppl1= 172.3468

NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 0 OOVs
0 zeroprobs, logprob= -215815.3 ppl= 6.908426 ppl1= 7.375255

PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 0 OOVs
0 zeroprobs, logprob= -130363.7 ppl= 174.8952 ppl1= 233.8374

SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 0 OOVs
0 zeroprobs, logprob= -100133.7 ppl= 129.0329 ppl1= 156.4847

UK
------------

In [17]:
#Modified KN Discount W/ interpolation


print("Modified KND with interpolation")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"{name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd_int.lm -ukndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd_int.lm")


Modified KND with interpolation
DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 0 OOVs
0 zeroprobs, logprob= -66618.8 ppl= 148.1737 ppl1= 174.9468

GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 0 OOVs
0 zeroprobs, logprob= -456812.4 ppl= 137.9866 ppl1= 170.232

NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 0 OOVs
0 zeroprobs, logprob= -213629.3 ppl= 6.774495 ppl1= 7.227485

PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 0 OOVs
0 zeroprobs, logprob= -134479.4 ppl= 205.8656 ppl1= 277.7808

SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 words, 0 OOVs
0 zeroprobs, logprob= -103669.6 ppl= 153.1906 ppl1= 187.0516


In [22]:
#BEST MODELS (per lang)

for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    elif name in ["DE","PL" ,"SP"]:
        print(f"{name} with no interpolation")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd.lm -ukndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd.lm")

    else:
        print(f"{name} with interpolation")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd_int.lm -ukndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd_int.lm")

DE with no interpolation
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 29702 words, 0 OOVs
0 zeroprobs, logprob= -64237.73 ppl= 123.9322 ppl1= 145.4591

GR with interpolation
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 8727 sentences, 204753 words, 0 OOVs
0 zeroprobs, logprob= -456812.4 ppl= 137.9866 ppl1= 170.232

NL with interpolation
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 248699 words, 0 OOVs
0 zeroprobs, logprob= -213629.3 ppl= 6.774495 ppl1= 7.227485

PL with no interpolation
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 55031 words, 0 OOVs
0 zeroprobs, logprob= -130363.7 ppl= 174.8952 ppl1= 233.8374

SP with no interpolation
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 45630 wo

In [34]:
#Testing the an to see their outputs
run_srilm("ngram -lm ../output/models/GR_fourgram_uknd_int.lm -seed 2026 -gen 100")

just think colleagues to talk case is that assign multiannual financial framework to stop and the agreement to have the courage to 135 eagerness large state so aegio kallikratis involved in the greek government policies
and after so many times the of logistics ships granting citizenship understand what is called for years now is the un a composite legislation 2009 see video photo gallery
why the the memorandum meeting even when compromises so wise yes we 2 which will help us
very soon change approximately battle and political will perform as of course when a knowledge based on this country much better demonstrates after the country in such they trap primary surpluses
therefore we at international level
and omissions of the economy now a unique opportunity increased feat citizens fighting rapporteur to open
is very recently products
it s estate
greek and physical out a vote of confidence in agreement with renewed the value
no time at the same years economic level eg for all greeks are c